# Retrieve NOAA NWM data

*Federal Open Water Data for Researchers* (CUAHSI), session 4.

**Learning goal:** Retrieve NOAA NWM data for a location of interest, describe what comes back using the course's shared concepts, and choose an access pattern that scales with your question.

**Environment:** run this notebook in the `m03-nwm` conda environment, created from `module-03_lesson-02_retrieve-noaa-nwm-data_environment.yml` in this folder (see the bundle README). Run the cells in order: later cells reuse variables from earlier ones.

**Student notebook.** "Now you try it" cells are starters: fill in the `...` blanks, guided by the `# TODO:` comments. The solutions notebook in the same folder has an example answer.

The NOAA National Water Model (NWM) simulates and forecasts streamflow for about 2.8 million river reaches across the United States. This lesson shows you how to find the reach you care about and retrieve its forecasts in Python. If you haven't met the NWM yet, start with Meet NOAA NWM, which covers how the model works, its configurations and its known limitations. You will retrieve NWM forecasts for the reach of the Ohio River at Louisville, Kentucky, both today's forecast and archived forecasts issued during the April 2025 flood.

## Attribution and licence

This notebook is part of *Federal Open Water Data for Researchers*, a CUAHSI course, and is released under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.html) (GPL-3.0). You may use, change and share it under the same licence.

It adapts code and ideas from these sources:

- Example code for `hydrotools` adapted from the [OWPHydroTools NWM Client README](https://github.com/NOAA-OWP/hydrotools/tree/main/python/nwm_client) (NOAA-OWP).

Other sources used in the lesson are listed under "Further reading" at the end of this notebook.

## Credentials

No account or key is needed for anything in this notebook: the NLDI, the NOAA NWM API and the NWM archives on Google Cloud and AWS are all public.

## Choosing an access route

NWM output is big: every forecast hour is a file covering all ~2.8 million reaches. How you should access it depends mostly on **how many forecast runs (issue times) you need**, and much less on how many reaches. Except for the CIROH BigQuery API (last row), none of these routes needs an API key.

| Use case | Recommended route | Key needed? | Scaling limits |
|---|---|---|---|
| Today's or the last few days' forecasts for a few reaches | NOAA NWM API | No | Keeps only ~3–5 days of runs; labelled experimental; built for a few reaches per request |
| Past forecasts, a few reaches, a few issue times | `hydrotools` ([Google Cloud archive](https://console.cloud.google.com/marketplace/product/noaa-public/national-water-model)) | No | Downloads whole CONUS (contiguous U.S.) files (~250–280 MB per short-range run); cost grows with the number of runs, not reaches |
| Past forecasts for many reaches or a region, or many issue times | Kerchunk references + `xarray`/`dask`, ideally run in the cloud | No | Reads only the `streamflow` chunk (~7× fewer bytes than whole files) with less memory, but each read is still a CONUS-wide chunk; building references is a one-time step per run |
| Long historical record (simulation, not forecasts) | NWM retrospective simulation (cloud Zarr) | No | A different product (see Temporal scaling) |
| Researchers working on CIROH projects who want to query selected forecast data without downloading whole files | [CIROH NWM BigQuery API](https://hub.ciroh.org/docs/products/data-management/bigquery-api/) | Yes, by request | You request only the data you need through a REST API backed by a BigQuery dataset, unlike `hydrotools`, which downloads whole files from the Google Cloud archive. Free for CIROH members and partners with active CIROH projects; request access and estimate query costs first (see the CIROH page). Not covered further in this lesson |

The numbers behind these recommendations are in Why NWM downloads cost what they cost.

## Tools and environment setup

This notebook runs in the `m03-nwm` conda environment, made from `module-03_lesson-02_retrieve-noaa-nwm-data_environment.yml` in this folder (the bundle README says how). Credentials are set up in the Credentials section above.

## Programmatic data discovery

Before you download anything, *discover* the reach (or reaches) you need. You don't want to page through the whole NWM domain to learn which COMID matches your location. Doing discovery first also lets you build a COMID list once and reuse it, which matters once you scale up to many reaches (see Spatial scaling).

**The point-and-click way.** NOAA's [National Water Prediction Service map](https://water.noaa.gov/map) lets you click a river and read off its reach ID, which is the same number as the COMID.

### Finding COMIDs with the NLDI

The NWM's *Location identifier* is the **COMID** of a reach (also called `feature_id`, `nwm_feature_id` or ComID; see the shared concepts in Meet NOAA NWM). The NLDI can:

* find the COMID for a point on the map (a lookup against NHDPlus catchments),
* find the COMID that a known monitoring site sits on, and
* navigate the river network upstream or downstream from a COMID (upstream mainstem, upstream with tributaries, downstream mainstem, downstream with diversions), which is useful for "every reach in this watershed" questions.

We call the NLDI from Python with the [`pynhd`](https://docs.hyriver.io/readme/pynhd.html) package (part of HyRiver).

**Example: what's the COMID of the Ohio River at Louisville?**

In [ ]:
from pynhd import NLDI

# the place we're asking about: the Ohio River at Louisville, KY (longitude, latitude; EPSG:4326)
POINT = (-85.7991, 38.2803)

nldi = NLDI()
comid_gdf = nldi.feature_byloc(POINT)   # snap the point to an NHDPlus reach
comid = int(comid_gdf["comid"].iloc[0])

print(comid_gdf.drop(columns="geometry"))
print(f"COMID: {comid}")

It returns a one-row `GeoDataFrame` with the matching `comid` and the reach's geometry (dropped from the printout). The Ohio at Louisville is COMID **10164004**.

The same lookup with the raw NLDI web address, which you can paste into a browser:

```text
https://api.water.usgs.gov/nldi/linked-data/comid/position?coords=POINT(-85.7991 38.2803)
```

**Example: the COMID of a reach with a monitoring site on it**

In [ ]:
# look up the reach by the ID of the monitoring site on it (used here only to locate the reach)
site = nldi.getfeature_byid("nwissite", "USGS-03294500")
print(site[["identifier", "name", "comid"]])

Both routes give the same COMID, which is a useful check that you found the right reach.

**Example: every reach upstream of Louisville, up to 100 km**

For regional questions you want a *list* of COMIDs. `navigate_byid` walks the network from a starting COMID. `upstreamTributaries` follows every tributary, and `distance` (in km) caps how far to go.

In [ ]:
# every NHDPlus reach upstream of the Louisville reach, tributaries included, up to 100 km away
upstream = nldi.navigate_byid(fsource="comid", fid=str(comid),
                              navigation="upstreamTributaries", source="flowlines", distance=100)
upstream_comids = upstream["nhdplus_comid"].astype(int).tolist()
print(f"{len(upstream_comids)} reaches upstream of (and including) the Louisville reach")

This returns a `GeoDataFrame` of reach lines with an `nhdplus_comid` column and a `geometry` column: **498 reaches** within 100 km upstream, including the Louisville reach itself. We reuse `upstream_comids` in Spatial scaling. On the Ohio, going farther upstream adds reaches quickly (the whole basin has tens of thousands), so set `distance` to what your question needs.

**Map: where these reaches are**

The geometries come back with the reaches, so the map needs no other data:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
upstream.plot(ax=ax, color="#9fb7d9", linewidth=0.8)                   # all 498 upstream reaches
comid_gdf.plot(ax=ax, color="#eb6834", linewidth=4)                    # the Louisville reach
ax.plot(*POINT, marker="*", color="#eb6834", markersize=16, markeredgecolor="white")
ax.annotate(f"COMID {comid}\nOhio River at Louisville, KY", POINT, xytext=(15, -45),
            textcoords="offset points", fontsize=10)
ax.set_aspect(1 / 0.786)                                               # roughly equal km in x and y at 38° N
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("NHDPlus reaches up to 100 km upstream of Louisville", loc="left", fontsize=12)
fig.tight_layout()
fig.savefig("nwm-ohio-louisville-map.png", dpi=150)

Once you have your COMID(s), move on to downloads.

## Programmatic data downloads

This section covers three routes, in order of how much data they're built for: the NOAA NWM API, `hydrotools`, and kerchunk references read with `xarray`. All three return the NWM's *Variable* `streamflow`, in m³/s (the API labels it `CMS`), for each COMID. The NWM has no per-value *Data quality flag(s)*, so you record the configuration, reference time and model version with your data instead (see Understanding what you downloaded).

Two terms you'll see in every route:

* **Configuration**: which forecast product, e.g. `short_range` (hourly out to 18 hours, issued every hour), `medium_range` (out to 10 days, issued every 6 hours, with several *ensemble members*: parallel runs from slightly different inputs, numbered by `member_id`, that show the spread of possible outcomes), `long_range`, or `analysis_assim` (the model's best estimate of current conditions).

* **Reference time**: model initialization time, in UTC (Coordinated Universal Time). NWM shorthand writes the reference hour with a Z for UTC: `00Z` is 00:00 UTC, and file names write it as `t00z`. The time each forecast value applies to is its **valid time** (`valid_datetime` in the API, `value_time` in `hydrotools`, `time` in the files).

### NOAA NWM API: today's forecasts for a few reaches

NOAA's [NWM API](https://api.water.noaa.gov/nwm/v1/docs) returns forecasts for one or more reaches as JSON (a text format that Python reads as lists and dictionaries), with no key and no files to manage. It is labelled **experimental**, and it keeps only roughly the **last 3–5 days** of forecasts (on 2026-10-09 the oldest short-range run it held was from 2026-10-05 00Z). Use it for "what is the model predicting now?", not for past events.

**Example: the latest short-range forecast for the Ohio at Louisville**

In [ ]:
import requests
import pandas as pd

comid = 10164004  # from the discovery step
url = f"https://api.water.noaa.gov/nwm/v1/streamflow/forecast/nwm_feature_id/{comid}/"

# ask for the most recent short-range forecast for this reach
resp = requests.get(url, params={"configuration": "short_range", "latest": "true"}, timeout=60)
resp.raise_for_status()                 # stop with an error if the request failed
run = resp.json()[0]                    # one entry per configuration + reference time

forecast = pd.DataFrame(run["forecast"][0]["timeseries"])
forecast["valid_datetime"] = pd.to_datetime(forecast["valid_datetime"])
print(run["configuration"], run["reference_datetime"], run["units"])
print(forecast.head())

Your numbers will differ: this is whatever run is newest when you ask. The response is a list with one entry per forecast run. Each entry has `nwm_feature_id` (the COMID), `configuration`, `reference_datetime`, `data_type`, `units` and a `forecast` list with one item per ensemble member (`member_id`). Each member holds a `timeseries` of `valid_datetime`/`value` pairs: 18 hourly values for a short-range run.

Leave out `latest` and use `min_reference_datetime` (and `max_reference_datetime`) to get several runs in one request:

In [ ]:
# every medium-range run issued in the last 2 days (the API only keeps the last few days)
since = (pd.Timestamp.now(tz="UTC") - pd.Timedelta("2D")).strftime("%Y-%m-%dT%H:00:00Z")
resp = requests.get(url, params={"configuration": "medium_range", "min_reference_datetime": since}, timeout=120)
resp.raise_for_status()

# flatten the nested JSON into one row per run, member and valid time
rows = []
for run in resp.json():
    for member in run["forecast"]:
        for step in member["timeseries"]:
            rows.append({"reference_time": run["reference_datetime"], "member": member["member_id"],
                         "valid_time": step["valid_datetime"], "streamflow_cms": step["value"]})

runs = pd.DataFrame(rows)
print(runs.groupby("reference_time")["member"].nunique())

`runs` is a long-format table (one row per value) with columns `reference_time`, `member`, `valid_time` and `streamflow_cms`. Each medium-range run had 6 ensemble members in the API response: member 1 runs out to 240 hours, the others to 204 hours. The whole response for seven runs was about 0.6 MB, and every short-range run the API held for this reach (99 runs) came back in one 0.13 MB response in under a second.

### `hydrotools`: past forecasts from the cloud archive

For anything older than a few days, such as the April 2025 flood, you need NOAA's archive of the raw NWM output files. The operational archive is mirrored, with no key or account needed, on [Google Cloud](https://console.cloud.google.com/marketplace/product/noaa-public/national-water-model) (`gs://national-water-model`) and on [AWS](https://registry.opendata.aws/noaa-nwm-pds/) (`s3://noaa-nwm-pds`). Each forecast hour is a separate NetCDF file covering all ~2.8 million reaches. A short-range channel_rt file (the channel-routing output, which holds `streamflow`) is about 13–16 MB.

* `hydrotools.nwm_client`: retrieves NWM streamflow forecasts. Older tutorials use `hydrotools.nwm_client_new`; its last release was 8.0.0 in December 2024, and development continues as [`hydrotools.nwm_client`](https://pypi.org/project/hydrotools.nwm-client/) (currently 9.x), with the same `NWMFileClient` interface.
* Canonical column names shared across the `hydrotools` subpackages: `value`, `value_time`, `variable_name`, `nwm_feature_id` (the COMID) and so on. Because every `hydrotools` table uses the same names, tables from different sources join easily.

In [ ]:
# In Jupyter only: importing hydrotools.nwm_client starts its own event loop, which fails inside the
# notebook's running loop. Import it once in a separate thread first. (nest_asyncio, suggested below,
# doesn't fix this with current packages; plain Python scripts need neither.)
import importlib
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(1) as pool:
    pool.submit(importlib.import_module, "hydrotools.nwm_client.NWMFileClient").result()

from concurrent.futures import ThreadPoolExecutor
from hydrotools.nwm_client.NWMFileClient import NWMFileClient

comid = 10164004  # from the discovery step

def get_forecast():
    # downloads go to ./hydrotools_data/NWMFileClient_NetCDF_files; results are cached in ./hydrotools_data/nwm_store.parquet
    client = NWMFileClient()   # default catalog: Google Cloud
    return client.get(
        configurations=["short_range"],
        reference_times=["2025-04-06T00:00"],   # forecast issue time (UTC)
        nwm_feature_ids=[comid],
    )

# hydrotools starts its own event loop to download files, which fails inside Jupyter's running loop.
# Running the call in its own thread avoids that, and it also works in a plain script.
with ThreadPoolExecutor(1) as pool:
    forecast_data = pool.submit(get_forecast).result()

print(forecast_data[["reference_time", "nwm_feature_id", "value_time", "value", "measurement_unit"]].head().to_string())

The result is a long-format `pandas.DataFrame`, one row per COMID and valid time (18 rows here), with columns `reference_time`, `nwm_feature_id`, `value_time`, `value`, `measurement_unit`, `variable_name`, `configuration` and `usgs_site_code` (a gage code that `hydrotools` fills in for reaches it maps to a stream gage; it is empty, `<NA>`, for this reach). Times are in UTC. Values are in SI units (`m3 s-1`); pass `unit_system=MeasurementUnitSystem.US` (from `hydrotools.nwm_client.NWMClientDefaults`) to `NWMFileClient` for cubic feet per second.

**If `.get()` fails:**

* **In Jupyter or another notebook**, `hydrotools` raises `RuntimeError: asyncio.run() cannot be called from a running event loop`, because its downloader starts its own event loop and the notebook already runs one. Run `import nest_asyncio; nest_asyncio.apply()` once before calling `.get()` (`nest-asyncio` is in the course environment). Plain Python scripts don't need it.
* **With errors about reading a NetCDF/HDF file**, an earlier run was probably interrupted (a timeout, a lost connection or a stopped cell), leaving a half-downloaded file in `hydrotools_data/`. `hydrotools` treats files already in that folder as downloaded, so it tries to open the broken one instead of fetching it again. Delete the `hydrotools_data/` folder in your working directory and run the call again.
* The `FutureWarning` and `UserWarning` messages printed during `.get()` (from `xarray`, `dask` and Google's libraries) are harmless: if the table prints, it worked.

### Kerchunk references: lazy, cloud-native reads with `xarray`

First, two terms. Inside a NetCDF file, each variable is stored in compressed blocks called *chunks*; a reader has to fetch and decompress a whole chunk to get any value in it, like having to take a whole box off the shelf to get one book. *Zarr* is a storage format that keeps each chunk as a separate object, so cloud tools can fetch exactly the chunks they need. NetCDF files aren't designed to be read piece by piece over the internet.

Two kinds of reference are available:

1. **NOAA-published references.** NOAA's Open Data Dissemination program publishes ready-made references in `s3://noaa-nodd-kerchunk-pds` ([registry entry](https://registry.opendata.aws/noaa-nodd-kerchunk/)) for NWM `short_range` and `medium_range_mem1` (plus the Alaska, Hawaii and Puerto Rico short range), built automatically as new files arrive. On 2026-10-07 they started at **2026-01-01**, so they don't cover the April 2025 flood.

2. **References you build yourself** for any files in the archive. This is a one-time step per forecast run, and the result is small (about 20 KB of JSON for an 18-file short-range run).

In [ ]:
import fsspec
import nwmurl
from concurrent.futures import ThreadPoolExecutor
from kerchunk.hdf import SingleHdf5ToZarr
from kerchunk.combine import MultiZarrToZarr

def index_one(url):
    """Scan one NetCDF file's internal layout and return its byte-range references."""
    with fsspec.open(url, "rb", block_size=2**20) as f:   # read in 1 MB blocks (see below)
        return SingleHdf5ToZarr(f, url, inline_threshold=500).translate()

def combine_refs(singles):
    """Stitch per-file references into one dataset along the time dimension.
    Runs in its own thread so it also works inside Jupyter."""
    def _combine():
        return MultiZarrToZarr(singles, remote_protocol="https", concat_dims=["time"],
                               identical_dims=["feature_id", "reference_time", "crs"]).translate()
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(_combine).result()

def build_refs(day, cycle, runinput=1, meminput=None, hours=18):
    """References for one forecast run: day "YYYYMMDD", cycle = issue hour (UTC)."""
    urls = nwmurl.generate_urls_operational(
        start_date=f"{day}0000", end_date=f"{day}0000",  # YYYYMMDDHHMM; one day
        fcst_cycle=[cycle],                 # issue time (UTC hour)
        lead_time=list(range(1, hours + 1)),  # forecast hours 1..hours
        varinput=1,                         # 1 = channel_rt (streamflow)
        geoinput=1,                         # 1 = CONUS
        runinput=runinput,                  # 1 = short_range, 2 = medium_range
        urlbaseinput=3,                     # 3 = https://storage.googleapis.com/national-water-model/
        meminput=meminput,                  # ensemble member (None for short_range)
    )
    with ThreadPoolExecutor(8) as pool:     # one request per file: run them in parallel
        singles = list(pool.map(index_one, urls))
    return combine_refs(singles)

refs = build_refs("20250406", 0)
print(len(refs["refs"]), "reference entries")

`refs` is a Python dictionary: a map from each chunk of each variable to a URL, byte offset and length. A few choices in this code are deliberate:

* `urlbaseinput=3` asks `nwmurl` for the public HTTPS address of each file in the Google Cloud archive, so both scanning and later reads use plain HTTPS with no cloud account. In our tests, reading through `gcsfs` with the current `zarr`/`kerchunk` versions stalled, while HTTPS worked. `nwmurl` can also point at the AWS copy (`urlbaseinput=7`) or other configurations (e.g. `runinput=2` and `meminput=1` for medium-range member 1, as in the figure below).

* `inline_threshold=500` stores tiny variables (like the single `time` value in each file) directly in the JSON, so the combine step doesn't need to download anything.
* `block_size=2**20` makes each file's scan one 1 MB request, which covers the file's metadata. fsspec's default 5 MB block read about 5× more data for no benefit, and smaller blocks took many more requests and were slower. In our test, building references for three runs took 16 s and 57 MB.
* Save `refs` to a JSON file (`json.dump(refs, open("refs_20250406_00z.json", "w"))`) and reuse it, rather than rebuilding.

Check that `nwmurl` gave you files that exist: a URL for a run that isn't in the archive fails when `index_one` opens it. `generate_urls_operational(..., enforce_valid_outputs=1)` checks each URL first and drops missing ones.

**Example: read the Louisville reach from the references**

In [ ]:
import xarray as xr

def open_refs(refs):
    """Open kerchunk references lazily: nothing is downloaded until values are needed."""
    return xr.open_dataset("reference://", engine="zarr", chunks={}, consolidated=False, zarr_format=2,
                           storage_options={"fo": refs, "remote_protocol": "https",
                                            "remote_options": {"asynchronous": True}})

ds = open_refs(refs)
print(ds)
print(ds.attrs["NWM_version_number"], ds.attrs["model_configuration"])

# pull one reach: reads 1 chunk per forecast hour (~2 MB each), not the whole files
q = ds["streamflow"].sel(feature_id=comid).load()
print(q.to_series().head())

`ds` is an `xarray.Dataset` with dimensions `time` (the 18 forecast hours, i.e. valid times) and `feature_id` (2,776,734 reaches), and variables `streamflow` (m³/s), `velocity`, `nudge` and a few runoff terms. Its attributes (`ds.attrs`) record the model version (`v3.0`) and configuration. The "2GB" size is what the full dataset *would* be in memory; nothing is downloaded until `.load()`. Note `chunksize=(1, 2776734)`: one chunk per hour holding every reach.

In [ ]:
import json
import fsspec
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from kerchunk.combine import MultiZarrToZarr

comid = 10164004                                # Ohio River at Louisville

s3 = fsspec.filesystem("s3", anon=True)        # public bucket: no AWS account needed

# NOAA publishes one reference file per forecast hour; pick yesterday's 12Z short-range run
day = (pd.Timestamp.now(tz="UTC") - pd.Timedelta("1D")).strftime("%Y%m%d")
paths = sorted(s3.glob(f"noaa-nodd-kerchunk-pds/nwm/short_range/"
                       f"nwm.short_range.channel_rt.conus.{day}.t12z.f*.nc.zarr"))

def load_ref(path):
    """Read one NOAA reference file and point it at the same NWM files over HTTPS."""
    text = json.dumps(json.loads(s3.cat(path)))
    return json.loads(text.replace("s3://noaa-nwm-pds/", "https://noaa-nwm-pds.s3.amazonaws.com/"))

refs_list = [load_ref(p) for p in paths]

def combine():
    return MultiZarrToZarr(refs_list, remote_protocol="https", concat_dims=["time"],
                           identical_dims=["feature_id", "reference_time", "crs"]).translate()

# Run the combine step in its own thread: kerchunk starts its own event loop,
# which fails inside Jupyter's running loop.
with ThreadPoolExecutor(1) as pool:
    recent = pool.submit(combine).result()

ds_recent = open_refs(recent)
print(len(paths), "reference files;", dict(ds_recent.sizes))
print(ds_recent["streamflow"].sel(feature_id=comid).load().to_series().head(3))

## Understanding what you downloaded

Whichever route you used, you now have NWM streamflow forecasts for one reach. Here is what each part means, using the shared concepts from Meet NOAA NWM:

| Shared concept | What it is in your download | Where to find it |
|---|---|---|
| Location identifier | COMID / `feature_id` `10164004` (the NHDPlus reach of the Ohio at Louisville) | `nwm_feature_id` (API, `hydrotools`), `feature_id` (files) |
| Variable | `streamflow`: modeled discharge through the reach | variable name in each route |
| Variable unit | m³/s | `units` = `CMS` (API), `measurement_unit` = `m3 s-1` (`hydrotools`), `ds["streamflow"].attrs["units"]` (files) |
| Time | reference time (model initialization time) and valid time (when each value applies), both UTC | `reference_datetime`/`valid_datetime` (API), `reference_time`/`value_time` (`hydrotools`), `reference_time`/`time` (files) |
| Data quality flag(s) | None. The NWM is a model, so there is no per-value flag. Judge forecasts by lead time (valid time minus reference time) and by comparing runs | – |
| Version / provenance | NWM v3.0, configuration `short_range` (or `medium_range` member 1), from the Google Cloud or AWS archive | `ds.attrs["NWM_version_number"]`, `ds.attrs["model_configuration"]`; the API's `configuration` field |
| Data unit | One NetCDF file per forecast hour, covering every reach; a short-range run is 18 files, a medium-range member 240 | file names in the archive (e.g. `nwm.t00z.short_range.channel_rt.f001.conus.nc`) |

**What the forecasts look like.** Each forecast run is a separate guess about the future, so a useful picture plots several runs together. The code below builds references for two more short-range runs and for medium-range member 1 issued on 3 April 2025 (240 files, about a minute to build), then plots them.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import StrMethodFormatter

# three short-range runs (18 h each) and one medium-range run (member 1, 240 h)
short_runs = {"2025-04-06 00Z": refs, "2025-04-06 12Z": build_refs("20250406", 12),
              "2025-04-07 00Z": build_refs("20250407", 0)}
medium = build_refs("20250403", 0, runinput=2, meminput=1, hours=240)

fig, ax = plt.subplots(figsize=(8, 4.2))
q_med = open_refs(medium)["streamflow"].sel(feature_id=comid).load().to_series()
ax.plot(q_med.index, q_med.values, color="#52514e", linewidth=2, label="Medium range, issued 2025-04-03 00Z (member 1)")
for (label, r), color in zip(short_runs.items(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    q_short = open_refs(r)["streamflow"].sel(feature_id=comid).load().to_series()
    ax.plot(q_short.index, q_short.values, color=color, linewidth=2, label=f"Short range, issued {label}")

ax.set_ylabel("Streamflow (m³/s)")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_xlabel("Valid time, 2025 (UTC)")
ax.set_title(f"NWM v3.0 forecasts, Ohio River at Louisville (COMID {comid})", loc="left", fontsize=12)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("nwm-ohio-louisville-forecasts.png", dpi=150)

If your plot looks like this, your code is right: these are model forecasts, not measurements, and this is what the archive holds (we've asked NOAA why the short-range runs drop so fast; see the note below). That disagreement is part of what you downloaded: the NWM gives you each run as issued, and different runs and configurations can tell different stories about the same river. Record exactly which runs you used, and look at several before drawing conclusions from one.

**Keep raw and derived data apart.** The files in the NOAA archive are your raw data, and they stay where they are.

## Best practices FAQs

The sections below answer these questions, with code examples. All three answers follow from how NWM files are stored, so we start with why NWM downloads cost what they cost.

* What is the recommended way to download data for **one location across the full period of record**? See Temporal scaling.
* What is the recommended way to download data across **all locations for a small time range**? See Spatial scaling.
* If I am working on improving efficiency through **code parallelization**, what should I do vs avoid? See Parallelization.

### Why NWM downloads cost what they cost

One fact about the files explains almost every recommendation below. Inside each NWM `channel_rt` file, `streamflow` is stored as **one compressed chunk per forecast hour that contains all 2,776,734 reaches** (about 2 MB compressed). There is no way to read "just my reach" from a file: any tool that reads the files has to fetch at least that whole chunk. A service such as the NOAA NWM API is different: it returns just the reaches you ask for, and whatever reading of the underlying data that takes happens on its side. For the files themselves:

* **Cost grows with the number of forecast hours (files) you touch, not with the number of reaches.**
* `hydrotools` downloads whole files (~13–16 MB each, every variable). Kerchunk references fetch only the `streamflow` chunk you need (~2 MB per hour), so they move fewer bytes, but the bytes are still CONUS-sized.

We measured this for the Louisville reach, using three short-range runs before the April 2025 crest (issued 6 April 00Z and 12Z and 7 April 00Z: 54 hourly files), from a home internet connection, on 2026-10-08:

| Route | Reaches | Data transferred | Peak memory | Wall time |
|---|---|---|---|---|
| NOAA NWM API (today's data; all 99 short-range runs it held) | 1 | 0.13 MB | 0.1 GB | < 1 s |
| `hydrotools` (Google Cloud) | 1 | 844 MB (saved to disk) | 1.7 GB | 2.6 min |
| `hydrotools` (Google Cloud) | 498 | 844 MB (saved to disk) | 1.8 GB | 2.7 min |
| Kerchunk references + `xarray` | 1 | 125 MB | 0.5 GB | 10 s |
| Kerchunk references + `xarray` | 498 | 125 MB | 1.0 GB* | 19 s |
| *Building the references (one time), as in the example above* | – | 57 MB, in 54 requests | 0.2 GB | 19 s |

One **medium-range** member (issued 2025-04-03 00Z: 240 hourly files) scales the same way, about 13× a short-range run:

| Route | Reaches | Data transferred | Peak memory | Wall time |
|---|---|---|---|---|
| `hydrotools` (Google Cloud) | 1 | ~3 GB or more (240 whole files); not re-measured, see the timeout note above | – | – |
| Kerchunk references + `xarray` | 1 | 520 MB | 0.4 GB | 38 s |
| *Building the references (one time)* | – | 252 MB, in 240 requests | 0.2 GB | 92 s |

### Temporal scaling

What is the recommended way to download data for one location but a long period?

**Recent forecasts (last few days):** use the NOAA NWM API. One request returns every run it still holds.

**Past forecasts (NWM forecast history):** every reference time is a separate set of files, so cost grows linearly with the number of forecast runs: ~280 MB per short-range run and ~3 GB per medium-range member with `hydrotools`, or ~40 MB per short-range run with kerchunk references.

### Spatial scaling

What is the recommended way to download data for all locations but a small time range?

Start with **discovery** to build your list of COMIDs (e.g. every reach up to 100 km upstream of Louisville, as above), then pass the whole list to the download step in one call rather than looping one COMID at a time. Looping over COMIDs re-reads the same CONUS-wide chunks for every reach.

* **`hydrotools`:** `NWMFileClient.get()` accepts a list of COMIDs. In our test, 498 reaches cost the same downloads and memory as 1 reach. If you omit `nwm_feature_ids`, it returns a default set: channel reaches that `hydrotools` maps to stream gages (8,866 in `hydrotools.nwm_client` 9.2.1).
* **Kerchunk references:** keep only the COMIDs that are NWM reaches (almost all NHDPlus reaches are, but check), then select them all at once (example below). Again, 498 reaches fetched exactly the same 125 MB as one reach.

* **NOAA NWM API:** accepts a comma-separated list of COMIDs, but it is built for a few reaches. For hundreds or thousands, use one of the file-based routes.

Here is the kerchunk version, reusing `ds` (the 6 April 00Z run) and `upstream_comids` from above:

In [ ]:
import numpy as np

# keep only COMIDs that exist in the NWM output, then read them all in one selection
ids = np.array(upstream_comids)
ids = ids[np.isin(ids, ds["feature_id"].values)]
basin = ds["streamflow"].sel(feature_id=ids).load()
print(basin.shape)   # (forecast hours, reaches)

`basin` is an `xarray.DataArray` with one row per forecast hour and one column per reach.

`hydrotools` results come back as pandas DataFrames that use categorical columns to save memory. The documentation notes that categorical columns can behave unexpectedly in groupby operations, are incompatible with fixed-format HDF files (use `format="table"`), and may cause problems when writing to geospatial formats with geopandas. Casting a categorical column to `str` resolves these issues.

### Parallelization

* **Parallelize over files (forecast hours or runs), not over reaches.** Reaches share chunks, so splitting COMIDs across workers multiplies the bytes; splitting files across workers doesn't.
* **Kerchunk + dask:** opening references with `chunks={}` gives you dask arrays with one task per chunk, so `.load()` fetches chunks concurrently. Building references makes one request per file (with 1 MB blocks), so a thread pool (as in the example above, 8 threads) helps a lot. We saw intermittent SSL errors from too many simultaneous connections at 16 threads (with `gcsfs`); fewer threads plus a retry fixed it.
* **`hydrotools`:** downloads are already asynchronous. The `FileDownloader` class has `limit` (default 10 concurrent downloads) and `timeout` (default 900 seconds) settings. Files for a single forecast cycle are processed in groups of 20 by default (the `group_size` parameter of `get_files()`); the documentation says this accommodates the xarray, dask, and HDF5 backends, which may struggle to open too many files at once, and that it matters mostly for medium-range forecasts. Running several `NWMFileClient.get()` calls at once mostly multiplies disk use and memory (~1.7 GB each in our test).
* **Run next to the data** when the job is big. Moving compute to a cloud machine in the bucket's region does more for large NWM jobs than any amount of local parallelism.

## Now you try it

Repeat the main steps for accessing NWM data on a second river: the **Willamette River at Salem, Oregon**. For helpful context, refer back to the example rivers we covered in the overview.

**Your task:** Find the COMID for the Willamette and retrieve short-range forecasts using the NWM API and Kerchunk references.

Compile relevant code from above to create a small, stand-alone code chunk that performs the steps below. Give the new results new names (for example, adding `_salem`), so you can still compare them with the Ohio River at Louisville.

1. Find the COMID of the Willamette at Salem with `nldi.feature_byloc`, by changing `POINT` to `(-123.0429, 44.9443)`.
2. Get the latest short-range forecast for that COMID from the NOAA NWM API. Change `comid` in the API example.
3. The Willamette ran high in late February 2026. Build kerchunk references for the short-range forecast issued at 12Z on 25 February 2026 with `build_refs("20260225", 12)` (about 20 seconds), then read the `streamflow` for your COMID with `open_refs`, as in the kerchunk example.

**What you should see:** COMID **23791093**; an API response with 18 hourly values in `CMS` (the API's `name` field is blank for this reach); and 18 published reference files for 25 February, with streamflow of about 1,700–1,800 m³/s, rising through the run.

In [ ]:
from pynhd import NLDI
import requests

POINT = ...   # TODO: (longitude, latitude) of the Willamette River at Salem, from the task
comid = ...   # TODO: snap POINT to an NHDPlus reach with NLDI().feature_byloc and keep its COMID as an int
print("COMID:", comid)

url = f"https://api.water.noaa.gov/nwm/v1/streamflow/forecast/nwm_feature_id/{comid}/"
resp = requests.get(url, params=..., timeout=60)   # TODO: ask for the latest short-range forecast
resp.raise_for_status()
run = resp.json()[0]
print(run["reference_datetime"], run["units"], len(run["forecast"][0]["timeseries"]), "values")

# Build references for the 2026-02-25 12Z short-range run from the Google Cloud archive
# (uses build_refs and open_refs from the kerchunk examples above)
refs_feb = build_refs(...)   # TODO: day "YYYYMMDD" and issue hour
q_feb = open_refs(refs_feb)["streamflow"].sel(feature_id=...).load().to_series()   # TODO
print(len(q_feb), "hourly values")
print(q_feb.iloc[[0, 8, 17]])

## Further reading

* Meet NOAA NWM (Module 2): what the model is and how its configurations differ.
* NLDI documentation: https://api.water.usgs.gov/docs/nldi
* `pynhd` (HyRiver) documentation: https://docs.hyriver.io/readme/pynhd.html; cite HyRiver as https://doi.org/10.21105/joss.03175
* NOAA NWM API (experimental) documentation: https://api.water.noaa.gov/nwm/v1/docs
* National Water Prediction Service API documentation: https://api.water.noaa.gov/nwps/v1/docs/
* OWPHydroTools (hydrotools) GitHub repo: https://github.com/NOAA-OWP/hydrotools
* OWPHydroTools NWM Client documentation: https://noaa-owp.github.io/hydrotools/hydrotools.nwm_client.html and package page: https://pypi.org/project/hydrotools.nwm-client/
* NWM operational archive on Google Cloud: https://console.cloud.google.com/marketplace/product/noaa-public/national-water-model
* NWM operational data on AWS (Registry of Open Data): https://registry.opendata.aws/noaa-nwm-pds/
* NOAA Cloud Optimized Zarr Reference Files (Kerchunk), Registry of Open Data: https://registry.opendata.aws/noaa-nodd-kerchunk/
* Kerchunk documentation: https://fsspec.github.io/kerchunk/
* `nwmurl` (CIROH), builds NWM file URLs: https://hub.ciroh.org/docs/products/data-management/dataaccess/NWMURL%20Library and source: https://github.com/CIROH-UA/nwmurl
* Tuhinanshu, T. (2023), *Using Kerchunk to make NOAA's National Water Model dataset more accessible*, Element 84: https://element84.com/software-engineering/using-kerchunk-to-make-noaas-national-water-model-dataset-more-accessible/
* NWM retrospective archive (Zarr, AWS): https://registry.opendata.aws/nwm-archive/
* Example code for `hydrotools` adapted from the [OWPHydroTools NWM Client README](https://github.com/NOAA-OWP/hydrotools/tree/main/python/nwm_client) (NOAA-OWP).